## Code to preprocess Chenchen Zhu's RNA Xenium Transcripts data using Voxels to create spatial bounding box

In [19]:
import gzip
import shutil
import pandas as pd
import numpy as np

# ----------------------------------
# 1. Unzip the CSV
# ----------------------------------
with gzip.open("data/16.csv.gz", "rb") as f_in:
    with open("data/16.csv", "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

# ----------------------------------
# 2. Load CSV
# ----------------------------------
df = pd.read_csv("data/16.csv")
print("Initial rows:", len(df))

# ----------------------------------
# FIX: Rename columns to ensure we have x, y, z, Cell Type
# ----------------------------------
# If there are exactly 4 columns, rename the 4th one (Xenium transcripts have last layer labelled differently)
if df.shape[1] == 4:
    df.columns = ['x', 'y', 'z', 'Cell Type']
else:
    raise ValueError(f"Expected 4 columns, found {df.shape[1]}. Columns = {df.columns.tolist()}")

print("Columns:", df.columns.tolist())


# ----------------------------------
# 3. Choose voxel size (downsampling rate)
# ----------------------------------
voxel_size = 3.6 

coords = df[['x', 'y', 'z']].to_numpy()
vox = np.floor(coords / voxel_size).astype(np.int32)

df['vx'] = vox[:, 0]
df['vy'] = vox[:, 1]
df['vz'] = vox[:, 2]

unique_vox = df[['vx','vy','vz']].drop_duplicates().shape[0]
print("Estimated unique voxels:", unique_vox)


# ----------------------------------
# 4. FAST voxel subsampling (sorted voxel grouping)
# ----------------------------------
vox_keys = df[['vx','vy','vz']].to_numpy()

# Sort by voxel key
order = np.lexsort((vox_keys[:,2], vox_keys[:,1], vox_keys[:,0]))
df_sorted = df.iloc[order].reset_index(drop=True)
coords_sorted = df_sorted[['x','y','z']].to_numpy()
vox_sorted = df_sorted[['vx','vy','vz']].to_numpy()

# Boundaries between voxel changes
changes = np.any(np.diff(vox_sorted, axis=0) != 0, axis=1)
boundaries = np.where(changes)[0] + 1
starts = np.concatenate(([0], boundaries))
ends = np.concatenate((boundaries, [len(df_sorted)]))

# Pick representative per voxel
picked_indices = []
for s, e in zip(starts, ends):
    g = coords_sorted[s:e]
    centroid = g.mean(axis=0)
    idx = np.linalg.norm(g - centroid, axis=1).argmin()
    picked_indices.append(s + idx)

sub_df = df_sorted.iloc[picked_indices].reset_index(drop=True)

print("Final subsampled rows:", len(sub_df))


# ----------------------------------
# 5. Save output
# ----------------------------------
out_path = "data/reduced_16.csv"
sub_df[['x','y','z','Cell Type']].to_csv(out_path, index=False)

print("Saved:", out_path)


Initial rows: 5197201
Columns: ['x', 'y', 'z', 'Cell Type']
Estimated unique voxels: 1045931
Final subsampled rows: 1045931
Saved: data/reduced_16.csv
